# Session 4 — Vectorized Math & the Grayscale Project

Runnable code for the **Build It** and **Experiment** sections. The image project
auto-generates a synthetic `profile.png` if you don't provide one, so it runs anywhere.

## 1. Build It — Core Code

In [1]:
import numpy as np

# ---------- 1. Vectorized math without loops ----------
a = np.array([1, 5, 2, 7])
print("Product of all elements:", np.prod(a))
print("Each element cubed:", np.power(a, 3))

# sine over a full range of angles at once
angles = np.linspace(-np.pi / 2, np.pi / 2, 100)
sines = np.sin(angles)
print("sin(-pi/2):", sines[0])   # expect close to -1
print("sin(pi/2): ", sines[-1])  # expect close to 1

# square root, real domain
print("sqrt(4):", np.sqrt(4))
print("sqrt(-4):", np.sqrt(-4))   # nan -- undefined in real numbers

Product of all elements: 70
Each element cubed: [  1 125   8 343]
sin(-pi/2): -1.0
sin(pi/2):  1.0
sqrt(4): 2.0
sqrt(-4): nan


C:\Users\panteaa\AppData\Local\Temp\ipykernel_2148\2045128960.py:16: RuntimeWarning: invalid value encountered in sqrt
  print("sqrt(-4):", np.sqrt(-4))   # nan -- undefined in real numbers


In [2]:
# ---------- 2. Complex-domain math with np.emath ----------
print("\nnp.emath.sqrt(-4):", np.emath.sqrt(-4))   # valid complex answer
print("np.log(-1):", np.log(-1))                    # nan
print("np.emath.log(-1):", np.emath.log(-1))        # valid complex answer
print("np.emath.sqrt(4) == np.sqrt(4)?", np.emath.sqrt(4) == np.sqrt(4))  # True for positive inputs


np.emath.sqrt(-4): 2j
np.log(-1): nan
np.emath.log(-1): 3.141592653589793j
np.emath.sqrt(4) == np.sqrt(4)? True


C:\Users\panteaa\AppData\Local\Temp\ipykernel_2148\1565093221.py:3: RuntimeWarning: invalid value encountered in log
  print("np.log(-1):", np.log(-1))                    # nan


In [3]:
# ---------- 3. Grayscale image conversion project ----------
import os
import numpy as np
from PIL import Image

image_path = "profile.png"

if not os.path.exists(image_path):
    # Create a small synthetic color image so the notebook runs anywhere
    h = w = 240
    r = np.linspace(0, 255, w, dtype=np.uint8).reshape(1, w)
    g = np.linspace(0, 255, h, dtype=np.uint8).reshape(h, 1)
    b = np.full((h, w), 128, dtype=np.uint8)
    synthetic = np.dstack([
        np.broadcast_to(r, (h, w)),
        np.broadcast_to(g, (h, w)),
        b,
    ])
    Image.fromarray(synthetic).save(image_path)
    print(f"No image found -- generated a synthetic placeholder at '{image_path}'")

original_img = Image.open(image_path)
img_data = np.asarray(original_img)
print("Image array shape:", img_data.shape)   # (H, W, 3)

No image found -- generated a synthetic placeholder at 'profile.png'
Image array shape: (240, 240, 3)


In [4]:
# Access individual color channels
red_channel = img_data[:, :, 0]
green_channel = img_data[:, :, 1]
blue_channel = img_data[:, :, 2]
print("Red channel mean/min/max:", red_channel.mean(), red_channel.min(), red_channel.max())

# Standard luminance-based grayscale formula, applied via matrix multiplication
weights = np.array([0.2126, 0.7152, 0.0722])
grayscale = img_data @ weights          # shape (H, W), floating point
grayscale = grayscale.astype(np.uint8)  # cast to valid 8-bit pixel values

gray_img = Image.fromarray(grayscale)
gray_img.save("profile_gray.png")
print("Saved grayscale image as profile_gray.png")

Red channel mean/min/max: 127.00416666666666 0 255
Saved grayscale image as profile_gray.png


## 2. Experiment

In [5]:
# Experiment 1: vectorized functions vs a manual loop -- same result, different speed
values = np.arange(1, 1_000_001)
vectorized_result = np.sqrt(values)
print("Vectorized sqrt of first 5:", vectorized_result[:5])

Vectorized sqrt of first 5: [1.         1.41421356 1.73205081 2.         2.23606798]


In [6]:
# Experiment 2: real vs complex sqrt/log across a mix of positive and negative numbers
test_values = np.array([4, -4, 9, -9, 16])
print("\nnp.sqrt (real domain):", np.sqrt(test_values.astype(float)))
print("np.emath.sqrt (complex-aware):", np.emath.sqrt(test_values))


np.sqrt (real domain): [ 2. nan  3. nan  4.]
np.emath.sqrt (complex-aware): [2.+0.j 0.+2.j 3.+0.j 0.+3.j 4.+0.j]


C:\Users\panteaa\AppData\Local\Temp\ipykernel_2148\886556567.py:3: RuntimeWarning: invalid value encountered in sqrt
  print("\nnp.sqrt (real domain):", np.sqrt(test_values.astype(float)))


In [7]:
# Experiment 3: grayscale weight sensitivity -- equal weights vs luminance weights
equal_weights = np.array([1 / 3, 1 / 3, 1 / 3])
gray_equal = (img_data @ equal_weights).astype(np.uint8)
print("Difference in mean brightness (luminance vs equal):",
      abs(float(grayscale.mean()) - float(gray_equal.mean())))
print("Human eyes are more sensitive to green, so luminance weighting looks more natural.")

Difference in mean brightness (luminance vs equal): 0.39661458333333144
Human eyes are more sensitive to green, so luminance weighting looks more natural.


In [8]:
# Experiment 4: what happens if you forget to cast to uint8?
gray_float = img_data @ weights
print("dtype before cast:", gray_float.dtype)
try:
    Image.fromarray(gray_float)
except Exception as e:
    print("Image.fromarray on float data ->", type(e).__name__, e)

dtype before cast: float64


## 3. Mini Project — Black-and-White Threshold Tool

In [9]:
# Reuse `grayscale` (uint8) from the Build It section
threshold = 128
bw = np.where(grayscale > threshold, 255, 0).astype(np.uint8)

Image.fromarray(bw).save("profile_bw.png")
print("Saved black-and-white image as profile_bw.png")
print("Percentage of white pixels:", (bw == 255).mean() * 100, "%")

Saved black-and-white image as profile_bw.png
Percentage of white pixels: 48.953125 %


In [10]:
# Experiment with different thresholds
for t in (100, 180):
    bw_t = np.where(grayscale > t, 255, 0).astype(np.uint8)
    print(f"threshold={t:>3} -> white pixels: {(bw_t == 255).mean() * 100:5.1f}%")

threshold=100 -> white pixels:  64.2%
threshold=180 -> white pixels:  20.5%


In [11]:
# Clean up generated image files so the repository stays clean
for f in ("profile.png", "profile_gray.png", "profile_bw.png"):
    if os.path.exists(f):
        os.remove(f)
print("Cleaned up generated images.")

Cleaned up generated images.
